In [ ]:
# ============================================================
# RSNA Knee Abnormality Detection — Stage 1: data contract
# ============================================================
# Reads only the five small competition CSVs. It never lists image
# directories recursively or prints identifiers or report text.

from pathlib import Path
import platform
import time
import uuid

import numpy as np
import pandas as pd
import pydicom
import sklearn
import torch

DATA_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ID_COLUMN = "StudyInstanceUID"
RUN_MODE = "full"  # one of: contract, smoke, full
RUN_NONCE = uuid.uuid4().hex
RUN_STARTED_AT = time.time()

required_files = {
    "train": DATA_ROOT / "train.csv",
    "train_series": DATA_ROOT / "train_series.csv",
    "test": DATA_ROOT / "test.csv",
    "test_series": DATA_ROOT / "test_series.csv",
    "sample_submission": DATA_ROOT / "sample_submission.csv",
}

# Load all five CSVs before asserting their contracts.
train_df = pd.read_csv(required_files["train"])
train_series_df = pd.read_csv(required_files["train_series"])
test_df = pd.read_csv(required_files["test"])
test_series_df = pd.read_csv(required_files["test_series"])
sample_submission_df = pd.read_csv(required_files["sample_submission"])

EXPECTED_TARGET_COLUMNS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]
TARGET_COLUMNS = [
    column for column in sample_submission_df.columns if column != ID_COLUMN
]

assert RUN_MODE in {"contract", "smoke", "full"}
assert all(
    ID_COLUMN in frame.columns
    for frame in (
        train_df,
        train_series_df,
        test_df,
        test_series_df,
        sample_submission_df,
    )
)
assert TARGET_COLUMNS == EXPECTED_TARGET_COLUMNS
assert len(TARGET_COLUMNS) == 12
assert list(sample_submission_df.columns) == [ID_COLUMN, *TARGET_COLUMNS]
assert set(TARGET_COLUMNS).issubset(train_df.columns)
assert train_df[ID_COLUMN].is_unique
assert test_df[ID_COLUMN].is_unique
assert sample_submission_df[ID_COLUMN].is_unique

print("Environment")
print(f"  Python: {platform.python_version()}")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA visible: {torch.cuda.is_available()}")
print("Competition metadata")
for name, frame in (
    ("train.csv", train_df),
    ("train_series.csv", train_series_df),
    ("test.csv", test_df),
    ("test_series.csv", test_series_df),
    ("sample_submission.csv", sample_submission_df),
):
    print(f"  {name}: rows={frame.shape[0]}, columns={frame.shape[1]}")
print(f"  targets: {len(TARGET_COLUMNS)}")
print("STAGE 1 PASSED: current-run contract and exact 12-target schema are valid")


In [ ]:
# ============================================================
# Report-derived supervision: high-precision weak labels
# ============================================================
# Reports remain in memory. This cell prints only aggregate diagnostics.

from dataclasses import dataclass
import re
import unicodedata


@dataclass(frozen=True)
class WeakLabel:
    state: str
    value: float
    weight: float
    conflict: bool = False


UNKNOWN = WeakLabel("unknown", np.nan, 0.0)
CONFIDENT_POSITIVE = WeakLabel("positive", 0.95, 0.70)
PROBABLE_POSITIVE = WeakLabel("positive", 0.80, 0.40)
CONFIDENT_NEGATIVE = WeakLabel("negative", 0.05, 0.50)

TARGET_CONCEPTS = {
    "ACL": [
        r"\bacl\b", r"\banterior cruciate ligament\b",
        r"\bligamento cruzado anterior\b", r"\blca\b",
    ],
    "MCL": [
        r"\bmcl\b", r"\bmedial collateral ligament\b",
        r"\bligamento colateral medial\b", r"\blcm\b",
    ],
    "Medial Meniscus": [
        r"\bmedial menisc(?:us|al)\b", r"\bmenisc(?:us|al)[ -]medial\b",
        r"\bmenisco medial\b", r"\bmenisque medial\b",
    ],
    "Lateral Meniscus": [
        r"\blateral menisc(?:us|al)\b", r"\bmenisc(?:us|al)[ -]lateral\b",
        r"\bmenisco lateral\b", r"\bmenisque lateral\b",
    ],
    "Medial OA": [
        r"\bmedial (?:compartment )?(?:oa|osteoarth(?:ritis|rosis)|arthrosis)\b",
        r"\b(?:oa|osteoarth(?:ritis|rosis)|arthrosis) (?:of (?:the )?)?medial compartment\b",
        r"\b(?:medial compartment|compartimento medial).{0,60}(?:degenerative change|joint space narrowing|chondrosis|cartilage loss|osteoartritis|artrosis)\b",
        r"\b(?:artrosis|osteoartritis) (?:del )?compartimento medial\b",
    ],
    "Lateral OA": [
        r"\blateral (?:compartment )?(?:oa|osteoarth(?:ritis|rosis)|arthrosis)\b",
        r"\b(?:oa|osteoarth(?:ritis|rosis)|arthrosis) (?:of (?:the )?)?lateral compartment\b",
        r"\b(?:lateral compartment|compartimento lateral).{0,60}(?:degenerative change|joint space narrowing|chondrosis|cartilage loss|osteoartritis|artrosis)\b",
        r"\b(?:artrosis|osteoartritis) (?:del )?compartimento lateral\b",
    ],
    "PF OA": [
        r"\bpatello?-?femoral (?:oa|osteoarth(?:ritis|rosis)|arthrosis|chondrosis|degeneration|cartilage loss)\b",
        r"\b(?:oa|osteoarth(?:ritis|rosis)|arthrosis) (?:of (?:the )?)?patello?-?femoral compartment\b",
        r"\b(?:artrosis|osteoartritis) patelofemoral\b",
    ],
    "Effusion": [
        r"\b(?:joint |articular )?effusion\b", r"\bderrame articular\b",
        r"\bgelenkerguss\b", r"\bepanchement articulaire\b",
    ],
    "Synovitis": [
        r"\bsynovitis\b", r"\bsinovitis\b", r"\bsynovial inflammation\b",
    ],
    "Baker's": [
        r"\bbaker'?s?[ -](?:cyst|cystic lesion)\b", r"\bpopliteal cyst\b",
        r"\bquiste (?:de )?baker\b", r"\bquiste popliteo\b",
        r"\bbaker[ -]zyste\b",
    ],
    "Contusion": [
        r"\b(?:marrow|bone|osseous) contusion\b", r"\bbone bruise\b",
        r"\bcontusion (?:medular|osea)\b", r"\bedema oseo traumatico\b",
        r"\btraumatic bone marrow edema\b",
    ],
    "Fracture": [
        r"\bfracture\b", r"\bfractura\b", r"\bfraktur\b",
        r"\b(?:cortical|trabecular) break\b",
    ],
}

COMPILED_TARGET_CONCEPTS = {
    target: re.compile("(?:" + "|".join(patterns) + ")")
    for target, patterns in TARGET_CONCEPTS.items()
}

NEGATION = re.compile(
    r"\b(?:no|not|without|absent|negative for|free of|neither|nor|"
    r"no evidence of|no sign of|sin|ausencia de|negativo para|no hay|"
    r"sem|kein|keine|pas de)\b"
)
UNCERTAINTY = re.compile(
    r"\b(?:cannot exclude|can(?:not|'t) rule out|not excluded|indeterminate|"
    r"equivocal|questionable|possible|possibly|may represent|could represent|"
    r"uncertain|cannot assess|limited evaluation|no se puede excluir|"
    r"indeterminado|dudoso|posible)\b"
)
HISTORY = re.compile(
    r"\b(?:status post|s/?p|history of|historical|previous|previously|prior|"
    r"remote|old|postoperative|postsurgical|repaired|reconstruction|repair|"
    r"antecedente de|estado posterior a|previamente|antigu[oa])\b"
)
PROBABLE = re.compile(
    r"\b(?:probable|probably|likely|most likely|favou?rs?|suspicious for|"
    r"consistent with|compatible with|presumed|presumptive|"
    r"probablemente|compatible con|sugestivo de)\b"
)
EXPLICIT_NEGATIVE = re.compile(
    r"\b(?:intact|preserved|unremarkable|normal|sin desgarro|integro|intacto)\b"
)
POSITIVE_FINDING = re.compile(
    r"\b(?:tear|torn|rupture|ruptured|sprain|injury|complete|partial|"
    r"degenerative|complex|radial|horizontal|flap|bucket-handle|"
    r"osteoarth(?:ritis|rosis)|arthrosis|oa|chondrosis|cartilage loss|"
    r"joint space narrowing|effusion|synovitis|cyst|contusion|bruise|"
    r"fracture|desgarro|rotura|artrosis|osteoartritis|derrame|sinovitis|"
    r"quiste|fractura|fraktur)\b"
)

CURRENT_FINDING = re.compile(
    r"\b(?:acute|new|current|recurrent|re-tear|retear|again|active|"
    r"agud[oa]|nuev[oa]|actual|recurrente)\b"
)
POST_NEGATION = re.compile(
    r"^\W*(?:is |are |appears? )?(?:absent|not seen|not present|intact|"
    r"preserved|unremarkable|normal|ausente|intacto|integro)\b"
)


def normalize_report(text: object) -> str:
    value = "" if pd.isna(text) else str(text)
    value = unicodedata.normalize("NFKD", value)
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = value.casefold().replace("’", "'")
    return re.sub(r"\s+", " ", value).strip()


def report_clauses_from_normalized(normalized: str) -> list[str]:
    return [
        clause.strip()
        for clause in re.split(
            r"(?:[.;:\n]+|,\s*(?:but|however|although|pero|sin embargo)\b|"
            r"\b(?:but|however|although|whereas|pero|sin embargo)\b)",
            normalized,
        )
        if clause.strip()
    ]


def trim_left_scope(text: str) -> str:
    parts = re.split(r"(?:,|\b(?:and|but|however|whereas|pero)\b)", text)
    return parts[-1] if parts else text


def trim_right_scope(text: str) -> str:
    return re.split(r"(?:,|\b(?:and|but|however|whereas|pero)\b)", text, maxsplit=1)[0]


def label_mention(clause: str, match: re.Match, radius: int = 72) -> WeakLabel:
    left = trim_left_scope(clause[max(0, match.start() - radius):match.start()])
    right = trim_right_scope(clause[match.end():match.end() + radius])
    local = f"{left} {match.group(0)} {right}"

    if UNCERTAINTY.search(local):
        return UNKNOWN
    if NEGATION.search(left) or EXPLICIT_NEGATIVE.search(left) or POST_NEGATION.search(right):
        return CONFIDENT_NEGATIVE
    if CURRENT_FINDING.search(local) and POSITIVE_FINDING.search(local):
        return CONFIDENT_POSITIVE
    if HISTORY.search(local):
        return UNKNOWN
    if PROBABLE.search(local) and POSITIVE_FINDING.search(local):
        return PROBABLE_POSITIVE
    if POSITIVE_FINDING.search(local):
        return CONFIDENT_POSITIVE
    return UNKNOWN


def aggregate_mentions(labels: list[WeakLabel]) -> WeakLabel:
    decisive_states = {label.state for label in labels if label.state != "unknown"}
    if len(decisive_states) > 1:
        return WeakLabel("unknown", np.nan, 0.0, True)
    if decisive_states:
        state = next(iter(decisive_states))
        return max(
            (label for label in labels if label.state == state),
            key=lambda label: label.weight,
        )
    return UNKNOWN


def analyze_report(text: object) -> dict[str, WeakLabel]:
    normalized = normalize_report(text)
    clauses = report_clauses_from_normalized(normalized)
    labels_by_target = {target: [] for target in TARGET_COLUMNS}
    for clause in clauses:
        for target, concept in COMPILED_TARGET_CONCEPTS.items():
            labels_by_target[target].extend(
                label_mention(clause, match) for match in concept.finditer(clause)
            )
    return {
        target: aggregate_mentions(labels)
        for target, labels in labels_by_target.items()
    }


def classify_target_mention(text: object, target: str) -> WeakLabel:
    if target not in COMPILED_TARGET_CONCEPTS:
        raise ValueError(f"Unsupported target: {target}")
    return analyze_report(text)[target]


PARSER_CASES = [
    ("complete acl tear", "ACL", "positive"),
    ("no acl tear", "ACL", "negative"),
    ("cannot exclude acl tear", "ACL", "unknown"),
    ("status post acl repair", "ACL", "unknown"),
    ("large joint effusion", "Effusion", "positive"),
    ("no joint effusion", "Effusion", "negative"),
    ("degenerative medial meniscus tear", "Medial Meniscus", "positive"),
    ("no acute fracture", "Fracture", "negative"),
    ("marrow contusion lateral femoral condyle", "Contusion", "positive"),
    ("small baker cyst", "Baker's", "positive"),
    ("no fracture but complete acl tear", "ACL", "positive"),
    ("no fracture but complete acl tear", "Fracture", "negative"),
    ("status post acl reconstruction with recurrent tear", "ACL", "positive"),
    ("cannot exclude medial meniscal tear; definite joint effusion", "Medial Meniscus", "unknown"),
    ("cannot exclude medial meniscal tear; definite joint effusion", "Effusion", "positive"),
    ("no acl or mcl tear", "ACL", "negative"),
    ("no acl or mcl tear", "MCL", "negative"),
    ("intact acl and medial meniscus tear", "ACL", "negative"),
    ("intact acl and medial meniscus tear", "Medial Meniscus", "positive"),
]
for report_text, target, expected_state in PARSER_CASES:
    parsed = classify_target_mention(report_text, target)
    assert parsed.state == expected_state, (target, expected_state, parsed.state)
assert len(TARGET_CONCEPTS) == len(TARGET_COLUMNS) == 12
assert set(TARGET_CONCEPTS) == set(TARGET_COLUMNS)
print(f"PARSER UNIT TESTS PASSED: {len(PARSER_CASES)}")

report_candidates = [
    column for column in train_df.columns
    if column.casefold() in {"report", "reporttext", "report_text"}
]
assert len(report_candidates) == 1, "Expected one report column"
REPORT_COLUMN = report_candidates[0]

weak_values = np.full((len(train_df), len(TARGET_COLUMNS)), np.nan, dtype=np.float32)
weak_weights = np.zeros_like(weak_values)
weak_conflicts = np.zeros_like(weak_weights, dtype=bool)

for row_position, report in enumerate(train_df[REPORT_COLUMN].tolist()):
    parsed_report = analyze_report(report)
    for target_position, target in enumerate(TARGET_COLUMNS):
        parsed = parsed_report[target]
        weak_values[row_position, target_position] = parsed.value
        weak_weights[row_position, target_position] = parsed.weight
        weak_conflicts[row_position, target_position] = parsed.conflict

official_values = train_df[TARGET_COLUMNS].apply(
    pd.to_numeric, errors="coerce"
).to_numpy(dtype=np.float32)
permanent_unlabeled = ~np.isfinite(official_values).any(axis=1)
assert permanent_unlabeled.any()
structural_enabled = np.zeros(len(TARGET_COLUMNS), dtype=bool)
parser_rows = []

for target_position, target in enumerate(TARGET_COLUMNS):
    covered_all = weak_weights[:, target_position] > 0
    covered = covered_all & permanent_unlabeled
    coverage = float(covered.sum() / permanent_unlabeled.sum())
    positive_prevalence = (
        float((weak_values[covered, target_position] >= 0.5).mean())
        if covered.any() else np.nan
    )
    gold = np.isfinite(official_values[:, target_position])
    gold_covered = gold & covered_all
    agreement = (
        float(
            ((weak_values[gold_covered, target_position] >= 0.5)
             == (official_values[gold_covered, target_position] >= 0.5)).mean()
        )
        if gold_covered.any() else np.nan
    )
    enabled = (
        0.01 <= coverage <= 0.98
        and np.isfinite(positive_prevalence)
        and 0.001 <= positive_prevalence <= 0.80
    )
    structural_enabled[target_position] = enabled
    parser_rows.append({
        "target": target,
        "coverage_pct": round(100 * coverage, 2),
        "positive_pct": round(100 * positive_prevalence, 2)
            if np.isfinite(positive_prevalence) else np.nan,
        "conflicts": int(weak_conflicts[:, target_position].sum()),
        "gold_compared": int(gold_covered.sum()),
        "agreement_pct_observational": round(100 * agreement, 2)
            if np.isfinite(agreement) else np.nan,
        "structural_enabled": bool(enabled),
    })


def fold_local_enabled_mask(train_gold_row_positions, min_agreement=0.60):
    train_gold_row_positions = np.asarray(train_gold_row_positions, dtype=np.int64)
    enabled = structural_enabled.copy()
    for target_position in range(len(TARGET_COLUMNS)):
        gold_values = official_values[train_gold_row_positions, target_position]
        parser_values = weak_values[train_gold_row_positions, target_position]
        parser_weight = weak_weights[train_gold_row_positions, target_position]
        compared = np.isfinite(gold_values) & (parser_weight > 0)
        if compared.any():
            agreement = float(
                ((parser_values[compared] >= 0.5) == (gold_values[compared] >= 0.5)).mean()
            )
            enabled[target_position] &= agreement >= min_agreement
    return enabled


gold_row_positions = np.flatnonzero(~permanent_unlabeled)
assert len(gold_row_positions) >= 2
fold_train_positions = gold_row_positions[:-1]
held_out_position = int(gold_row_positions[-1])
mask_before_heldout_change = fold_local_enabled_mask(fold_train_positions)
saved_values = weak_values[held_out_position].copy()
saved_weights = weak_weights[held_out_position].copy()
weak_values[held_out_position] = np.where(np.isfinite(saved_values), 1.0 - saved_values, 0.95)
weak_weights[held_out_position] = 0.70
mask_after_heldout_change = fold_local_enabled_mask(fold_train_positions)
weak_values[held_out_position] = saved_values
weak_weights[held_out_position] = saved_weights
assert np.array_equal(mask_before_heldout_change, mask_after_heldout_change)

supervision_values = weak_values.copy()
supervision_weights = weak_weights.copy()
supervision_weights[:, ~structural_enabled] = 0.0
supervision_values[:, ~structural_enabled] = np.nan
gold_mask = np.isfinite(official_values)
supervision_values[gold_mask] = official_values[gold_mask]
supervision_weights[gold_mask] = 1.0

assert np.array_equal(supervision_values[gold_mask], official_values[gold_mask])
assert np.isfinite(supervision_values[supervision_weights > 0]).all()
assert (supervision_weights >= 0).all() and (supervision_weights <= 1).all()
assert not np.isfinite(supervision_values[supervision_weights == 0]).any()

parser_summary = pd.DataFrame(parser_rows)
print(parser_summary.to_string(index=False))
print(
    "REPORT SUPERVISION PASSED: "
    f"12 targets checked, {int(structural_enabled.sum())} structurally enabled; gold agreement is diagnostic only"
)


In [ ]:
# ============================================================
# Deterministic multi-plane MRI sampling and DICOM normalization
# ============================================================

from collections import Counter
from PIL import Image

PLANE_TO_INDEX = {"Sagittal": 0, "Coronal": 1, "Axial": 2}
PLANE_NORMALIZATION = {name.casefold(): name for name in PLANE_TO_INDEX}


class InvalidDicomSlice(RuntimeError):
    pass


class UnsupportedMultiframe(RuntimeError):
    pass


def uniform_positions(length: int, count: int) -> np.ndarray:
    if length <= 0 or count <= 0:
        raise ValueError("length and count must be positive")
    low = 0.05 * (length - 1)
    high = 0.95 * (length - 1)
    return np.rint(np.linspace(low, high, count)).astype(np.int64).clip(0, length - 1)


def natural_name_key(name: str):
    return tuple(
        (0, int(part)) if part.isdigit() else (1, part.casefold())
        for part in re.split(r"(\d+)", str(name))
        if part
    )


def flag_value(value: object) -> int:
    return int(str(value).strip().casefold() in {"1", "1.0", "true", "yes", "y"})


def orientation_normal(header) -> np.ndarray | None:
    orientation = getattr(header, "ImageOrientationPatient", None)
    if orientation is None:
        return None
    values = np.asarray(orientation, dtype=np.float64)
    if values.size < 6 or not np.isfinite(values[:6]).all():
        return None
    normal = np.cross(values[:3], values[3:6])
    magnitude = float(np.linalg.norm(normal))
    return normal / magnitude if np.isfinite(magnitude) and magnitude > 0 else None


def normals_consistent(normals: list[np.ndarray], tolerance: float = 0.999) -> bool:
    if not normals:
        return False
    reference = normals[0]
    return all(abs(float(np.dot(reference, normal))) >= tolerance for normal in normals[1:])


def mri_ordered_paths(series_directory: Path, diagnostics: Counter | None = None) -> list[Path]:
    diagnostics = diagnostics if diagnostics is not None else Counter()
    records = []
    for path in series_directory.glob("*.dcm"):
        try:
            header = pydicom.dcmread(path, stop_before_pixels=True, force=False)
            rows = int(getattr(header, "Rows", 0) or 0)
            columns = int(getattr(header, "Columns", 0) or 0)
            frames = int(getattr(header, "NumberOfFrames", 1) or 1)
            if rows <= 0 or columns <= 0:
                raise InvalidDicomSlice("Missing image dimensions")
            if frames != 1:
                diagnostics["multiframe_rejected"] += 1
                continue
            position = getattr(header, "ImagePositionPatient", None)
            position = np.asarray(position, dtype=np.float64) if position is not None else None
            if position is not None and (position.size < 3 or not np.isfinite(position[:3]).all()):
                position = None
            normal = orientation_normal(header)
            instance_raw = getattr(header, "InstanceNumber", None)
            try:
                instance = float(instance_raw) if instance_raw is not None else None
                if instance is not None and not np.isfinite(instance):
                    instance = None
            except (TypeError, ValueError):
                instance = None
            records.append({
                "path": path,
                "position": position,
                "normal": normal,
                "instance": instance,
                "sop": str(getattr(header, "SOPInstanceUID", "")),
                "natural": natural_name_key(path.name),
            })
        except Exception:
            diagnostics["header_failures"] += 1

    if not records:
        raise InvalidDicomSlice("No usable single-frame DICOM headers")

    spatial_complete = all(
        record["position"] is not None and record["normal"] is not None
        for record in records
    )
    normals = [record["normal"] for record in records if record["normal"] is not None]
    if spatial_complete and normals_consistent(normals):
        reference = normals[0]
        records.sort(key=lambda record: (
            float(np.dot(record["position"][:3], reference)),
            record["natural"],
            record["sop"],
        ))
    elif all(record["instance"] is not None for record in records):
        diagnostics["spatial_fallbacks"] += 1
        records.sort(key=lambda record: (
            record["instance"], record["natural"], record["sop"],
        ))
    else:
        diagnostics["spatial_fallbacks"] += 1
        records.sort(key=lambda record: (record["natural"], record["sop"]))
    return [record["path"] for record in records]


def series_quality_profile(series_directory: Path) -> tuple[int, float]:
    diagnostics = Counter()
    try:
        paths = mri_ordered_paths(series_directory, diagnostics)
        header = pydicom.dcmread(paths[len(paths) // 2], stop_before_pixels=True, force=False)
        rows = int(getattr(header, "Rows", 0) or 0)
        columns = int(getattr(header, "Columns", 0) or 0)
        spacing = np.asarray(getattr(header, "PixelSpacing", [1.0, 1.0]), dtype=np.float64)
        if spacing.size < 2 or not np.isfinite(spacing[:2]).all() or (spacing[:2] <= 0).any():
            spacing = np.array([1.0, 1.0])
        coverage = float(rows * spacing[0] * columns * spacing[1])
        return len(paths), coverage
    except Exception:
        return 0, 0.0


def choose_plane_series(
    rows: pd.DataFrame,
    image_root: Path | None = None,
    study_id: str | None = None,
) -> pd.DataFrame:
    required = {
        "SeriesInstanceUID", "Anatomical_Plane",
        "Fluid_Sensitive", "Fat_Suppression",
    }
    if not required.issubset(rows.columns):
        raise ValueError(f"Missing series columns: {sorted(required - set(rows.columns))}")
    ranked = rows.copy()
    ranked["_plane"] = ranked["Anatomical_Plane"].astype(str).str.strip().str.casefold().map(
        PLANE_NORMALIZATION
    )
    ranked = ranked[ranked["_plane"].notna()].copy()
    ranked["_plane_rank"] = ranked["_plane"].map(PLANE_TO_INDEX)
    ranked["_fluid"] = ranked["Fluid_Sensitive"].map(flag_value)
    ranked["_fat"] = ranked["Fat_Suppression"].map(flag_value)
    ranked["_series_key"] = ranked["SeriesInstanceUID"].astype(str)
    if image_root is not None and study_id is not None:
        profiles = [
            series_quality_profile(image_root / str(study_id) / series_id)
            for series_id in ranked["_series_key"]
        ]
        ranked["_usable"] = [profile[0] for profile in profiles]
        ranked["_coverage"] = [profile[1] for profile in profiles]
        ranked = ranked[ranked["_usable"] > 0].copy()
    else:
        ranked["_usable"] = 0
        ranked["_coverage"] = 0.0
    ranked = ranked.sort_values(
        ["_plane_rank", "_fluid", "_fat", "_usable", "_coverage", "_series_key"],
        ascending=[True, False, False, False, False, True],
        kind="stable",
    )
    chosen = ranked.drop_duplicates("_plane_rank", keep="first").copy()
    chosen["Anatomical_Plane"] = chosen["_plane"]
    return chosen.sort_values("_plane_rank", kind="stable").drop(
        columns=[
            "_plane", "_plane_rank", "_fluid", "_fat", "_series_key",
            "_usable", "_coverage",
        ]
    )


def normalize_mri_pixels(image: np.ndarray, photometric: str) -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    if image.ndim != 2:
        raise UnsupportedMultiframe("Only single-frame 2-D DICOM slices are supported")
    finite_mask = np.isfinite(image)
    if not finite_mask.any():
        raise InvalidDicomSlice("Slice has no finite pixels")
    fill_value = float(np.median(image[finite_mask]))
    image = np.where(finite_mask, image, fill_value).astype(np.float32)
    if str(photometric).strip().upper() == "MONOCHROME1":
        image = float(image.max() + image.min()) - image
    low, high = np.percentile(image, [1.0, 99.0])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        raise InvalidDicomSlice("Slice has no usable intensity range")
    return np.clip((image - low) / (high - low), 0.0, 1.0).astype(np.float32)


def physical_square_resize(
    image: np.ndarray,
    pixel_spacing: np.ndarray,
    image_size: int = 224,
) -> np.ndarray:
    if image_size <= 0:
        raise ValueError("image_size must be positive")
    spacing = np.asarray(pixel_spacing, dtype=np.float64)
    if spacing.size < 2 or not np.isfinite(spacing[:2]).all() or (spacing[:2] <= 0).any():
        spacing = np.array([1.0, 1.0])
    height, width = image.shape
    physical_height = height * float(spacing[0])
    physical_width = width * float(spacing[1])
    scale = max(height, width) / max(physical_height, physical_width)
    resized_height = max(1, int(round(physical_height * scale)))
    resized_width = max(1, int(round(physical_width * scale)))
    physical = Image.fromarray(np.rint(image * 255).astype(np.uint8)).resize(
        (resized_width, resized_height), Image.Resampling.BICUBIC
    )
    physical_array = np.asarray(physical, dtype=np.float32) / 255.0
    side = max(resized_height, resized_width)
    canvas = np.zeros((side, side), dtype=np.float32)
    top = (side - resized_height) // 2
    left = (side - resized_width) // 2
    canvas[top:top + resized_height, left:left + resized_width] = physical_array
    output = np.asarray(
        Image.fromarray(np.rint(canvas * 255).astype(np.uint8)).resize(
            (image_size, image_size), Image.Resampling.BICUBIC
        ),
        dtype=np.float32,
    ) / 255.0
    if output.shape != (image_size, image_size) or not np.isfinite(output).all():
        raise InvalidDicomSlice("Invalid resized MRI slice")
    return output


def decode_mri_slice(path: Path, image_size: int = 224) -> np.ndarray:
    try:
        dataset = pydicom.dcmread(path, force=False)
        frames = int(getattr(dataset, "NumberOfFrames", 1) or 1)
        if frames != 1:
            raise UnsupportedMultiframe("Enhanced multiframe DICOM is not supported")
        if "PixelData" not in dataset:
            raise InvalidDicomSlice("DICOM has no pixel data")
        rows = int(getattr(dataset, "Rows", 0) or 0)
        columns = int(getattr(dataset, "Columns", 0) or 0)
        if rows <= 0 or columns <= 0:
            raise InvalidDicomSlice("DICOM has invalid image dimensions")
        pixels = np.asarray(dataset.pixel_array, dtype=np.float32)
        if pixels.ndim != 2:
            raise UnsupportedMultiframe("Decoded pixels are not a 2-D slice")
        pixels = pixels * float(getattr(dataset, "RescaleSlope", 1.0))
        pixels = pixels + float(getattr(dataset, "RescaleIntercept", 0.0))
        normalized = normalize_mri_pixels(
            pixels, getattr(dataset, "PhotometricInterpretation", "MONOCHROME2")
        )
        spacing = np.asarray(getattr(dataset, "PixelSpacing", [1.0, 1.0]), dtype=np.float64)
        return physical_square_resize(normalized, spacing, image_size)
    except (InvalidDicomSlice, UnsupportedMultiframe):
        raise
    except Exception as error:
        raise InvalidDicomSlice("DICOM decode failed") from error


def sample_series_slices(
    series_directory: Path,
    count: int = 8,
    image_size: int = 224,
) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    diagnostics = Counter()
    paths = mri_ordered_paths(series_directory, diagnostics)
    requested = uniform_positions(len(paths), count)
    decoded_cache = {}
    failed_indices = set()
    slices = []
    actual_positions = []

    for requested_position in requested:
        candidates = sorted(
            range(len(paths)),
            key=lambda index: (abs(index - int(requested_position)), index),
        )
        selected = None
        for index in candidates:
            if index in failed_indices:
                continue
            if index not in decoded_cache:
                try:
                    decoded_cache[index] = decode_mri_slice(paths[index], image_size)
                except UnsupportedMultiframe:
                    diagnostics["multiframe_rejected"] += 1
                    failed_indices.add(index)
                    continue
                except InvalidDicomSlice:
                    diagnostics["decode_failures"] += 1
                    failed_indices.add(index)
                    continue
            selected = decoded_cache[index]
            actual_positions.append(index)
            break
        if selected is None:
            raise InvalidDicomSlice("No decodable 2-D slices in selected series")
        slices.append(selected)

    stacked = np.stack(slices, axis=0).astype(np.float32)
    if stacked.shape != (count, image_size, image_size):
        raise InvalidDicomSlice("Unexpected sampled MRI shape")
    if not np.isfinite(stacked).all() or stacked.min() < 0 or stacked.max() > 1:
        raise InvalidDicomSlice("Sampled MRI values must be finite in [0, 1]")
    return stacked, np.asarray(actual_positions, dtype=np.int64), dict(diagnostics)


# Deterministic unit and edge-case tests.
assert uniform_positions(30, 8).tolist() == [1, 5, 9, 13, 16, 20, 24, 28]
assert uniform_positions(3, 8).shape == (8,)
assert set(uniform_positions(3, 8).tolist()) <= {0, 1, 2}
assert sorted(["10.dcm", "2.dcm", "1.dcm"], key=natural_name_key) == [
    "1.dcm", "2.dcm", "10.dcm"
]
assert normals_consistent([
    np.array([1.0, 0.0, 0.0]), np.array([-1.0, 0.0, 0.0])
])
assert not normals_consistent([
    np.array([1.0, 0.0, 0.0]), np.array([0.0, 1.0, 0.0])
])

synthetic_series = pd.DataFrame({
    "SeriesInstanceUID": ["sag-low", "sag-best", "cor", "ax"],
    "Anatomical_Plane": ["Sagittal", "sagittal", "Coronal", "Axial"],
    "Fluid_Sensitive": [0, 1, 1, 0],
    "Fat_Suppression": [0, 1, 0, 1],
})
chosen_synthetic = choose_plane_series(synthetic_series)
assert chosen_synthetic["Anatomical_Plane"].tolist() == ["Sagittal", "Coronal", "Axial"]
assert chosen_synthetic["SeriesInstanceUID"].tolist()[0] == "sag-best"

synthetic_pixels = np.arange(64, dtype=np.float32).reshape(8, 8)
mono2 = normalize_mri_pixels(synthetic_pixels, "MONOCHROME2")
mono1 = normalize_mri_pixels(synthetic_pixels, "MONOCHROME1")
assert np.allclose(mono1, 1.0 - mono2, atol=1e-6)
for invalid_pixels in (
    np.full((4, 4), np.nan, dtype=np.float32),
    np.ones((4, 4), dtype=np.float32),
    np.ones((2, 4, 4), dtype=np.float32),
):
    try:
        normalize_mri_pixels(invalid_pixels, "MONOCHROME2")
        raise AssertionError("Invalid pixels were accepted")
    except (InvalidDicomSlice, UnsupportedMultiframe):
        pass

# Corrupt/empty-series handling must fail with aggregate counters and redacted errors.
temporary_bad_dir = Path("/kaggle/temp") / f"mri_invalid_{RUN_NONCE}"
temporary_bad_dir.mkdir(parents=True, exist_ok=True)
temporary_bad_file = temporary_bad_dir / "10.dcm"
temporary_bad_file.write_bytes(b"not-a-dicom")
bad_diagnostics = Counter()
try:
    mri_ordered_paths(temporary_bad_dir, bad_diagnostics)
    raise AssertionError("Corrupt series was accepted")
except InvalidDicomSlice as error:
    assert str(temporary_bad_dir) not in str(error)
    assert bad_diagnostics["header_failures"] == 1
finally:
    temporary_bad_file.unlink(missing_ok=True)
    temporary_bad_dir.rmdir()

TRAIN_IMAGE_ROOT = DATA_ROOT / "train_series"
plane_counts = train_series_df.groupby(ID_COLUMN)["Anatomical_Plane"].nunique()
real_study_id = str(plane_counts.idxmax())
real_rows = train_series_df[train_series_df[ID_COLUMN].astype(str) == real_study_id]
real_choice = choose_plane_series(real_rows, TRAIN_IMAGE_ROOT, real_study_id)
assert 1 <= len(real_choice) <= 3
real_series_id = str(real_choice.iloc[0]["SeriesInstanceUID"])
real_directory = TRAIN_IMAGE_ROOT / real_study_id / real_series_id
real_slices, real_positions, real_diagnostics = sample_series_slices(
    real_directory, count=2, image_size=224
)
assert real_slices.shape == (2, 224, 224)
assert real_positions.shape == (2,)
assert set(real_diagnostics).issubset({
    "header_failures", "multiframe_rejected", "spatial_fallbacks", "decode_failures"
})
print(
    "MRI SAMPLING PASSED: "
    f"planes={len(real_choice)}, decoded_slices={len(real_slices)}, "
    f"failures={sum(real_diagnostics.values())}, shape=224x224"
)


In [ ]:
# ============================================================
# Frozen DINOv2 ViT-S/14 feature extraction
# ============================================================
# Kaggle Model: metaresearch/dinov2, PyTorch small V1, Apache 2.0.

import json
import os
from collections import Counter

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
from transformers import AutoImageProcessor, AutoModel

DINO_MODEL_DIR = Path(
    "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
)
assert DINO_MODEL_DIR.is_dir(), "DINOv2 small V1 model is not attached"
assert (DINO_MODEL_DIR / "config.json").is_file()
assert (DINO_MODEL_DIR / "preprocessor_config.json").is_file()
assert (DINO_MODEL_DIR / "pytorch_model.bin").is_file()


def select_dino_backend(
    cuda_available: bool,
    capability: tuple[int, int] | None = None,
    compiled_arches: tuple[str, ...] | list[str] = (),
) -> str:
    if not cuda_available or capability is None:
        return "cpu"
    architecture = f"sm_{capability[0]}{capability[1]}"
    return "cuda" if architecture in set(compiled_arches) else "cpu"


def inspect_dino_backend() -> bool:
    if not torch.cuda.is_available():
        return False
    return select_dino_backend(
        True,
        torch.cuda.get_device_capability(0),
        list(torch.cuda.get_arch_list()),
    ) == "cuda"


assert select_dino_backend(True, (6, 0), ["sm_70", "sm_75", "sm_80"]) == "cpu"
assert select_dino_backend(True, (7, 5), ["sm_70", "sm_75", "sm_80"]) == "cuda"
USE_DINO_CUDA = inspect_dino_backend()
if RUN_MODE == "full" and not USE_DINO_CUDA:
    raise RuntimeError("Full DINOv2 extraction requires a compatible T4 CUDA device")
DINO_DEVICE = torch.device("cuda" if USE_DINO_CUDA else "cpu")
DINO_GPU_CAPABILITIES = [
    tuple(torch.cuda.get_device_capability(index))
    for index in range(torch.cuda.device_count())
] if USE_DINO_CUDA else []
DINO_GPU_NAMES = [
    torch.cuda.get_device_name(index)
    for index in range(torch.cuda.device_count())
] if USE_DINO_CUDA else []
if USE_DINO_CUDA:
    assert DINO_GPU_CAPABILITIES
    assert all(capability == (7, 5) for capability in DINO_GPU_CAPABILITIES)
    assert all("T4" in name.upper() for name in DINO_GPU_NAMES)
    assert all(
        select_dino_backend(True, capability, list(torch.cuda.get_arch_list())) == "cuda"
        for capability in DINO_GPU_CAPABILITIES
    )

# Pin the official slow processor and verify its complete disk contract.
processor_disk_config = json.loads(
    (DINO_MODEL_DIR / "preprocessor_config.json").read_text(encoding="utf-8")
)
assert processor_disk_config["do_resize"] is True
assert processor_disk_config["size"] == {"shortest_edge": 256}
assert processor_disk_config["do_center_crop"] is True
assert processor_disk_config["crop_size"] == {"height": 224, "width": 224}
assert processor_disk_config["do_rescale"] is True
assert np.isclose(processor_disk_config["rescale_factor"], 1.0 / 255.0)
assert processor_disk_config["do_normalize"] is True
assert int(processor_disk_config["resample"]) == int(Image.Resampling.BICUBIC)
dino_processor = AutoImageProcessor.from_pretrained(
    DINO_MODEL_DIR,
    local_files_only=True,
    use_fast=False,
)
assert np.allclose(dino_processor.image_mean, [0.485, 0.456, 0.406])
assert np.allclose(dino_processor.image_std, [0.229, 0.224, 0.225])
assert int(dino_processor.crop_size["height"]) == 224
assert int(dino_processor.crop_size["width"]) == 224
assert int(dino_processor.resample) == int(Image.Resampling.BICUBIC)


def grayscale_to_rgb_uint8(grayscale_slices: np.ndarray) -> list[np.ndarray]:
    slices = np.asarray(grayscale_slices, dtype=np.float32)
    if slices.ndim != 3 or slices.shape[1:] != (224, 224):
        raise ValueError(f"Expected [N,224,224] grayscale input, received {slices.shape}")
    if not np.isfinite(slices).all() or slices.min() < 0 or slices.max() > 1:
        raise ValueError("DINOv2 inputs must be finite grayscale values in [0, 1]")
    grayscale_u8 = np.rint(slices * 255).astype(np.uint8)
    rgb = np.repeat(grayscale_u8[..., None], 3, axis=-1)
    assert np.array_equal(rgb[..., 0], rgb[..., 1])
    assert np.array_equal(rgb[..., 1], rgb[..., 2])
    return [image for image in rgb]


def prepare_dino_batch(grayscale_slices: np.ndarray) -> torch.Tensor:
    processed = dino_processor(
        images=grayscale_to_rgb_uint8(grayscale_slices),
        return_tensors="pt",
    )
    pixel_values = processed["pixel_values"]
    if pixel_values.ndim != 4 or pixel_values.shape[1:] != (3, 224, 224):
        raise ValueError(f"Unexpected DINOv2 tensor shape: {tuple(pixel_values.shape)}")
    if not torch.isfinite(pixel_values).all():
        raise ValueError("DINOv2 processor produced non-finite values")
    return pixel_values


processor_test = np.stack(
    [np.zeros((224, 224), dtype=np.float32), np.ones((224, 224), dtype=np.float32)],
    axis=0,
)
processor_test_values = prepare_dino_batch(processor_test)
assert processor_test_values.shape == (2, 3, 224, 224)
expected_black = -torch.tensor(dino_processor.image_mean) / torch.tensor(
    dino_processor.image_std
)
expected_white = (1.0 - torch.tensor(dino_processor.image_mean)) / torch.tensor(
    dino_processor.image_std
)
assert torch.allclose(processor_test_values[0, :, 112, 112], expected_black, atol=1e-5)
assert torch.allclose(processor_test_values[1, :, 112, 112], expected_white, atol=1e-5)
processor_mean = torch.tensor(dino_processor.image_mean).view(1, 3, 1, 1)
processor_std = torch.tensor(dino_processor.image_std).view(1, 3, 1, 1)
denormalized_test = processor_test_values * processor_std + processor_mean
assert torch.allclose(denormalized_test[:, 0], denormalized_test[:, 1], atol=2e-6)
assert torch.allclose(denormalized_test[:, 1], denormalized_test[:, 2], atol=2e-6)

# Load entirely from the attached Kaggle model with internet disabled.
dino_model = AutoModel.from_pretrained(
    DINO_MODEL_DIR,
    local_files_only=True,
)
assert dino_model.config.model_type == "dinov2"
assert int(dino_model.config.hidden_size) == 384
assert int(dino_model.config.num_channels) == 3
assert int(dino_model.config.patch_size) == 14
dino_model.eval()
for parameter in dino_model.parameters():
    parameter.requires_grad_(False)
assert not any(parameter.requires_grad for parameter in dino_model.parameters())
dino_model.to(DINO_DEVICE)

# DataParallel accelerates only the large feature batches; CPU and one-GPU paths stay simple.
dino_encoder = (
    torch.nn.DataParallel(dino_model)
    if USE_DINO_CUDA and torch.cuda.device_count() > 1
    else dino_model
)
dino_encoder.eval()


def extract_dino_embeddings(
    grayscale_slices: np.ndarray,
    batch_size: int = 32,
) -> np.ndarray:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")
    batches = []
    with torch.inference_mode():
        for start in range(0, len(grayscale_slices), batch_size):
            pixel_values = prepare_dino_batch(
                grayscale_slices[start:start + batch_size]
            ).to(DINO_DEVICE, non_blocking=USE_DINO_CUDA)
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_DINO_CUDA,
            ):
                output = dino_encoder(pixel_values=pixel_values)
                embeddings = output.last_hidden_state[:, 0, :]
            batches.append(embeddings.float().cpu())
    if not batches:
        raise ValueError("No slices supplied for DINOv2 extraction")
    result = torch.cat(batches, dim=0).numpy().astype(np.float16)
    if result.shape != (len(grayscale_slices), 384):
        raise ValueError(f"Unexpected DINOv2 embedding shape: {result.shape}")
    if not np.isfinite(result).all():
        raise ValueError("DINOv2 produced non-finite embeddings")
    return result


# Sample exactly two real slices from every selected anatomical plane.
smoke_slice_batches = []
smoke_plane_indices = []
smoke_diagnostics = Counter()
for _, series_row in real_choice.iterrows():
    plane_name = str(series_row["Anatomical_Plane"]).strip()
    assert plane_name in PLANE_TO_INDEX
    plane_index = PLANE_TO_INDEX[plane_name]
    series_directory = (
        TRAIN_IMAGE_ROOT
        / real_study_id
        / str(series_row["SeriesInstanceUID"])
    )
    plane_slices, _, plane_diagnostics = sample_series_slices(
        series_directory,
        count=2,
        image_size=224,
    )
    assert plane_slices.shape == (2, 224, 224)
    smoke_slice_batches.append(plane_slices)
    smoke_plane_indices.extend([plane_index] * len(plane_slices))
    smoke_diagnostics.update(plane_diagnostics)

real_multiplane_slices = np.concatenate(smoke_slice_batches, axis=0)
real_plane_indices = np.asarray(smoke_plane_indices, dtype=np.int8)
assert real_multiplane_slices.shape == (2 * len(real_choice), 224, 224)
assert real_plane_indices.shape == (len(real_multiplane_slices),)
assert np.isin(real_plane_indices, list(PLANE_TO_INDEX.values())).all()
assert Counter(real_plane_indices.tolist()) == Counter({
    PLANE_TO_INDEX[str(row["Anatomical_Plane"]).strip()]: 2
    for _, row in real_choice.iterrows()
})

real_embeddings = extract_dino_embeddings(real_multiplane_slices, batch_size=6)
assert real_embeddings.shape == (len(real_plane_indices), 384)
assert real_embeddings.dtype == np.float16
assert len(real_embeddings) == len(real_plane_indices)

DINO_CACHE_ROOT = Path("/kaggle/working") / f"cache_{RUN_NONCE}" / "dino"
DINO_CACHE_ROOT.mkdir(parents=True, exist_ok=False)
smoke_embedding_path = DINO_CACHE_ROOT / "smoke_embeddings.npy"
smoke_plane_path = DINO_CACHE_ROOT / "smoke_plane_indices.npy"
np.save(smoke_embedding_path, real_embeddings, allow_pickle=False)
np.save(smoke_plane_path, real_plane_indices, allow_pickle=False)
reloaded_embeddings = np.load(smoke_embedding_path, allow_pickle=False)
reloaded_plane_indices = np.load(smoke_plane_path, allow_pickle=False)
assert np.array_equal(reloaded_embeddings, real_embeddings)
assert np.array_equal(reloaded_plane_indices, real_plane_indices)
assert len(reloaded_embeddings) == len(reloaded_plane_indices)
print(
    "DINO FEATURE TEST PASSED: "
    f"planes={len(real_choice)}, instances={len(real_embeddings)}, "
    f"dimension=384, cuda={int(USE_DINO_CUDA)}, "
    f"gpus={len(DINO_GPU_CAPABILITIES)}, "
    f"capabilities={DINO_GPU_CAPABILITIES or ["cpu"]}, cache_verified=1"
)


In [ ]:
# ============================================================
# Weighted target-query attention MIL head
# ============================================================

import math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F


class TargetAttentionMIL(nn.Module):
    def __init__(
        self,
        feature_dim: int = 384,
        num_planes: int = 3,
        num_targets: int = 12,
        token_dim: int = 192,
        plane_dim: int = 32,
        hidden_dim: int = 96,
        dropout: float = 0.20,
    ):
        super().__init__()
        if not 0 < plane_dim < token_dim:
            raise ValueError("plane_dim must be smaller than token_dim")
        self.feature_dim = int(feature_dim)
        self.num_planes = int(num_planes)
        self.num_targets = int(num_targets)
        image_dim = token_dim - plane_dim

        self.feature_projection = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, image_dim),
            nn.GELU(),
        )
        self.plane_embedding = nn.Embedding(num_planes, plane_dim)
        self.token_fusion = nn.Sequential(
            nn.Linear(token_dim, token_dim),
            nn.GELU(),
            nn.LayerNorm(token_dim),
        )
        self.target_queries = nn.Parameter(torch.empty(num_targets, token_dim))
        nn.init.trunc_normal_(self.target_queries, std=0.02)
        self.attention_scale = token_dim ** -0.5
        self.head_trunk = nn.Sequential(
            nn.LayerNorm(token_dim),
            nn.Linear(token_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.output_weight = nn.Parameter(torch.empty(num_targets, hidden_dim))
        self.output_bias = nn.Parameter(torch.zeros(num_targets))
        nn.init.xavier_uniform_(self.output_weight)

    def forward(
        self,
        features: torch.Tensor,
        plane_indices: torch.Tensor,
        slice_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        if features.ndim != 3 or features.shape[-1] != self.feature_dim:
            raise ValueError("features must have shape [batch, slices, feature_dim]")
        if plane_indices.shape != features.shape[:2]:
            raise ValueError("plane_indices must align one-to-one with slice features")
        if plane_indices.dtype not in (torch.int32, torch.int64):
            raise ValueError("plane_indices must be an integer tensor")
        if torch.any(plane_indices < 0) or torch.any(plane_indices >= self.num_planes):
            raise ValueError("plane_indices contain an invalid anatomical plane")

        if slice_mask is None:
            slice_mask = torch.ones(
                features.shape[:2], dtype=torch.bool, device=features.device
            )
        if slice_mask.shape != features.shape[:2] or slice_mask.dtype != torch.bool:
            raise ValueError("slice_mask must be boolean with shape [batch, slices]")
        if not torch.all(slice_mask.any(dim=1)):
            raise ValueError("every study must contain at least one valid slice")

        image_tokens = self.feature_projection(features.float())
        plane_tokens = self.plane_embedding(plane_indices.long())
        tokens = self.token_fusion(torch.cat([image_tokens, plane_tokens], dim=-1))

        queries = F.layer_norm(
            self.target_queries,
            normalized_shape=(self.target_queries.shape[-1],),
        )
        attention_logits = torch.einsum(
            "bsd,td->bts", tokens, queries
        ) * self.attention_scale
        attention_logits = attention_logits.masked_fill(
            ~slice_mask[:, None, :], torch.finfo(attention_logits.dtype).min
        )
        attention_weights = torch.softmax(attention_logits, dim=-1)
        pooled_targets = torch.einsum("bts,bsd->btd", attention_weights, tokens)
        target_hidden = self.head_trunk(pooled_targets)
        logits = torch.einsum(
            "bth,th->bt", target_hidden, self.output_weight
        ) + self.output_bias
        if logits.shape != (features.shape[0], self.num_targets):
            raise RuntimeError("unexpected MIL output shape")
        return logits


def weighted_masked_bce(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weights: torch.Tensor,
    pos_weight: torch.Tensor | None = None,
) -> torch.Tensor:
    if logits.shape != targets.shape or logits.shape != weights.shape:
        raise ValueError("logits, targets, and weights must have identical shapes")
    if torch.any(torch.isfinite(weights) & (weights < 0)):
        raise ValueError("finite supervision weights cannot be negative")
    if pos_weight is not None:
        if pos_weight.ndim != 1 or pos_weight.shape[0] != logits.shape[-1]:
            raise ValueError("pos_weight must have one value per target")
        if not torch.all(torch.isfinite(pos_weight)) or torch.any(pos_weight <= 0):
            raise ValueError("pos_weight values must be finite and positive")
        pos_weight = pos_weight.to(device=logits.device, dtype=logits.dtype)
    known = torch.isfinite(targets) & torch.isfinite(weights) & (weights > 0)
    safe_targets = torch.where(known, targets, torch.zeros_like(targets))
    if torch.any((safe_targets[known] < 0) | (safe_targets[known] > 1)):
        raise ValueError("known targets must be probabilities in [0, 1]")
    safe_weights = torch.where(known, weights, torch.zeros_like(weights))
    elementwise = F.binary_cross_entropy_with_logits(
        logits, safe_targets, reduction="none", pos_weight=pos_weight
    )
    numerator = (elementwise * safe_weights).sum()
    denominator = safe_weights.sum()
    return numerator / denominator.clamp_min(
        torch.finfo(denominator.dtype).eps
    )


# -------------------- executable contract tests --------------------
torch.manual_seed(20260808)
test_model = TargetAttentionMIL(dropout=0.0)
test_features = torch.randn(4, 7, 384)
test_planes = torch.tensor(
    [[0, 0, 1, 1, 2, 2, 2]] * 4, dtype=torch.long
)
test_mask = torch.tensor(
    [[True, True, True, True, True, True, False]] * 4
)
test_logits = test_model(test_features, test_planes, test_mask)
assert test_logits.shape == (4, 12)
assert torch.isfinite(test_logits).all()

try:
    bad_planes = test_planes.clone()
    bad_planes[0, 0] = 3
    test_model(test_features, bad_planes, test_mask)
    raise AssertionError("invalid plane index was accepted")
except ValueError as exc:
    assert "invalid anatomical plane" in str(exc)

fixed_logits = torch.randn(3, 12, requires_grad=True)
base_targets = torch.randint(0, 2, (3, 12), dtype=torch.float32)
base_weights = torch.ones(3, 12)
base_weights[0, 4] = 0.0
loss_a = weighted_masked_bce(fixed_logits, base_targets, base_weights)
mutated_targets = base_targets.clone()
mutated_targets[0, 4] = float("nan")
mutated_weights = base_weights.clone()
mutated_weights[0, 4] = float("nan")
loss_b = weighted_masked_bce(fixed_logits, mutated_targets, mutated_weights)
assert torch.equal(loss_a.detach(), loss_b.detach())

# Exact gold/weak confidence-weight regression using the BCE closed form.
weight_logits = torch.tensor([[0.2, -0.7, 1.1, -1.3]])
weight_targets = torch.tensor([[1.0, 0.8, 0.05, 0.0]])
confidence_weights = torch.tensor([[1.0, 0.70, 0.40, 0.50]])
weighted_loss = weighted_masked_bce(
    weight_logits, weight_targets, confidence_weights
)
closed_form_bce = (
    torch.clamp(weight_logits, min=0)
    - weight_logits * weight_targets
    + torch.log1p(torch.exp(-torch.abs(weight_logits)))
)
expected_weighted_loss = (
    closed_form_bce * confidence_weights
).sum() / confidence_weights.sum()
assert torch.allclose(weighted_loss, expected_weighted_loss, atol=1e-7, rtol=0)

# Fold-specific target positive weights are accepted and shape-validated.
unit_pos_weight = torch.ones(4)
assert torch.allclose(
    weighted_masked_bce(
        weight_logits, weight_targets, confidence_weights, unit_pos_weight
    ),
    weighted_loss,
    atol=1e-7,
    rtol=0,
)

unknown_logits = torch.randn(2, 12, requires_grad=True)
unknown_loss = weighted_masked_bce(
    unknown_logits,
    torch.full((2, 12), float("nan")),
    torch.zeros(2, 12),
)
assert unknown_loss.requires_grad and float(unknown_loss.detach()) == 0.0
unknown_loss.backward()
assert torch.equal(unknown_logits.grad, torch.zeros_like(unknown_logits))

torch.manual_seed(7)
fit_features = torch.randn(10, 9, 384)
fit_planes = torch.tensor([[0, 0, 0, 1, 1, 1, 2, 2, 2]] * 10)
teacher = torch.randn(384, 12)
fit_targets = ((fit_features.mean(dim=1) @ teacher) > 0).float()
fit_weights = torch.ones_like(fit_targets)
fit_model = TargetAttentionMIL(dropout=0.0)
fit_optimizer = torch.optim.AdamW(fit_model.parameters(), lr=3e-3, weight_decay=0.0)
with torch.no_grad():
    initial_fit_loss = float(weighted_masked_bce(
        fit_model(fit_features, fit_planes), fit_targets, fit_weights
    ))
first_step_fit_loss = None
for step in range(24):
    fit_optimizer.zero_grad(set_to_none=True)
    fit_loss = weighted_masked_bce(
        fit_model(fit_features, fit_planes), fit_targets, fit_weights
    )
    fit_loss.backward()
    fit_optimizer.step()
    if step == 0:
        with torch.no_grad():
            first_step_fit_loss = float(weighted_masked_bce(
                fit_model(fit_features, fit_planes), fit_targets, fit_weights
            ))
assert first_step_fit_loss is not None
assert first_step_fit_loss <= initial_fit_loss
with torch.no_grad():
    final_fit_loss = float(weighted_masked_bce(
        fit_model(fit_features, fit_planes), fit_targets, fit_weights
    ))
assert np.isfinite(final_fit_loss)
assert final_fit_loss < initial_fit_loss * 0.80

print(
    "TARGET-ATTENTION MIL PASSED: "
    f"logits={tuple(test_logits.shape)}, "
    f"masked_loss=valid, overfit={initial_fit_loss:.4f}->{final_fit_loss:.4f}"
)


In [ ]:
# ============================================================
# Cross-validation, run-private feature cache, and fold training
# ============================================================
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
from collections import Counter
import os
import time

from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

FOLD_SEED = 20260808
N_FOLDS = 5
FULL_RUNTIME_ABORT_HOURS = 7.5
gold_mask = train_df[TARGET_COLUMNS].notna().any(axis=1).to_numpy()
gold_positions = np.flatnonzero(gold_mask)
gold_rows = train_df.iloc[gold_positions].reset_index(drop=True)
gold_y = gold_rows[TARGET_COLUMNS].to_numpy(dtype=np.float32)
assert len(gold_rows) == 58
assert np.isfinite(gold_y).all()
assert np.isin(gold_y, [0.0, 1.0]).all()

PATIENT_COLUMN_CANDIDATES = (
    "PatientID", "PatientId", "Patient_ID", "patient_id", "SubjectID"
)
patient_column = next(
    (column for column in PATIENT_COLUMN_CANDIDATES if column in gold_rows.columns),
    None,
)


def greedy_group_multilabel_folds(
    labels: np.ndarray,
    groups: np.ndarray,
    n_folds: int,
    seed: int,
) -> np.ndarray:
    unique_groups, group_codes = np.unique(groups.astype(str), return_inverse=True)
    group_labels = np.zeros((len(unique_groups), labels.shape[1]), dtype=np.float64)
    group_sizes = np.zeros(len(unique_groups), dtype=np.int64)
    for group_index in range(len(unique_groups)):
        members = group_codes == group_index
        group_labels[group_index] = labels[members].max(axis=0)
        group_sizes[group_index] = int(members.sum())

    positive_totals = group_labels.sum(axis=0).clip(min=1.0)
    rarity = (group_labels / positive_totals).sum(axis=1)
    rng = np.random.default_rng(seed)
    tie_noise = rng.random(len(unique_groups)) * 1e-9
    order = np.lexsort((tie_noise, -group_sizes, -rarity))

    fold_positive = np.zeros((n_folds, labels.shape[1]), dtype=np.float64)
    fold_sizes = np.zeros(n_folds, dtype=np.int64)
    group_fold = np.full(len(unique_groups), -1, dtype=np.int64)
    desired_positive = group_labels.sum(axis=0) / n_folds
    desired_size = len(labels) / n_folds
    base_capacity = len(labels) // n_folds
    fold_capacity = np.full(n_folds, base_capacity, dtype=np.int64)
    fold_capacity[: len(labels) % n_folds] += 1

    for group_index in order:
        costs = []
        for fold_index in range(n_folds):
            if fold_sizes[fold_index] + group_sizes[group_index] > fold_capacity[fold_index]:
                continue
            positive_after = fold_positive[fold_index] + group_labels[group_index]
            positive_cost = np.mean(
                ((positive_after - desired_positive) / np.maximum(desired_positive, 1.0)) ** 2
            )
            size_after = fold_sizes[fold_index] + group_sizes[group_index]
            size_cost = ((size_after - desired_size) / max(desired_size, 1.0)) ** 2
            costs.append((positive_cost + 0.20 * size_cost, fold_sizes[fold_index], fold_index))
        if not costs:
            raise RuntimeError("group sizes cannot satisfy deterministic fold capacities")
        chosen_fold = min(costs)[2]
        group_fold[group_index] = chosen_fold
        fold_positive[chosen_fold] += group_labels[group_index]
        fold_sizes[chosen_fold] += group_sizes[group_index]

    assignment = group_fold[group_codes]
    if np.any(assignment < 0):
        raise RuntimeError("fold assignment is incomplete")
    return assignment


fold_method = "greedy-group-multilabel"
if patient_column is None:
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

        splitter = MultilabelStratifiedKFold(
            n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED
        )
        fold_assignment = np.full(len(gold_rows), -1, dtype=np.int64)
        for fold_index, (_, validation_index) in enumerate(
            splitter.split(np.zeros(len(gold_rows)), gold_y)
        ):
            fold_assignment[validation_index] = fold_index
        fold_method = "iterative-multilabel-study"
    except (ImportError, ModuleNotFoundError):
        fold_assignment = greedy_group_multilabel_folds(
            gold_y,
            gold_rows[ID_COLUMN].astype(str).to_numpy(),
            N_FOLDS,
            FOLD_SEED,
        )
        fold_method = "greedy-multilabel-study"
else:
    fold_assignment = greedy_group_multilabel_folds(
        gold_y,
        gold_rows[patient_column].astype(str).to_numpy(),
        N_FOLDS,
        FOLD_SEED,
    )

assert np.all((fold_assignment >= 0) & (fold_assignment < N_FOLDS))
folds = [
    (
        np.flatnonzero(fold_assignment != fold_index),
        np.flatnonzero(fold_assignment == fold_index),
    )
    for fold_index in range(N_FOLDS)
]

seen_validation = set()
validation_sizes = []
for fold_index, (training_index, validation_index) in enumerate(folds):
    train_ids = set(gold_rows.iloc[training_index][ID_COLUMN].astype(str))
    valid_ids = set(gold_rows.iloc[validation_index][ID_COLUMN].astype(str))
    assert train_ids.isdisjoint(valid_ids)
    assert seen_validation.isdisjoint(valid_ids)
    seen_validation.update(valid_ids)
    validation_sizes.append(len(validation_index))
    if patient_column is not None:
        train_patients = set(gold_rows.iloc[training_index][patient_column].astype(str))
        valid_patients = set(gold_rows.iloc[validation_index][patient_column].astype(str))
        assert train_patients.isdisjoint(valid_patients)
assert len(seen_validation) == len(gold_rows)
assert max(validation_sizes) - min(validation_sizes) <= 2


def fold_positive_weight(
    targets: np.ndarray,
    weights: np.ndarray,
    minimum: float = 0.50,
    maximum: float = 4.00,
) -> torch.Tensor:
    known = np.isfinite(targets) & np.isfinite(weights) & (weights > 0)
    safe_targets = np.where(known, targets, 0.0)
    safe_weights = np.where(known, weights, 0.0)
    positive_mass = (safe_weights * safe_targets).sum(axis=0)
    negative_mass = (safe_weights * (1.0 - safe_targets)).sum(axis=0)
    both_classes = (positive_mass > 1e-6) & (negative_mass > 1e-6)
    ratios = np.ones_like(positive_mass, dtype=np.float64)
    ratios[both_classes] = (
        negative_mass[both_classes] / positive_mass[both_classes]
    )
    ratios = np.clip(ratios, minimum, maximum).astype(np.float32)
    if not np.isfinite(ratios).all() or np.any(ratios <= 0):
        raise RuntimeError("invalid fold positive weights")
    return torch.from_numpy(ratios)


_pos_weight_test = fold_positive_weight(
    np.array([[1.0, 0.0], [0.0, 0.0]], dtype=np.float32),
    np.ones((2, 2), dtype=np.float32),
)
assert torch.allclose(_pos_weight_test, torch.tensor([1.0, 1.0]))


class CachedFeatureDataset(Dataset):
    def __init__(
        self,
        features: np.ndarray,
        planes: np.ndarray,
        masks: np.ndarray,
        targets: np.ndarray,
        weights: np.ndarray,
        row_indices: np.ndarray,
    ):
        self.features = features
        self.planes = planes
        self.masks = masks
        self.targets = targets
        self.weights = weights
        self.row_indices = np.asarray(row_indices, dtype=np.int64)
        if not (
            features.shape[:2] == planes.shape == masks.shape
            and len(features) == len(targets) == len(weights)
        ):
            raise ValueError("cached feature and supervision arrays are misaligned")

    def __len__(self):
        return len(self.row_indices)

    def __getitem__(self, index):
        row = int(self.row_indices[index])
        mask = self.masks[row].astype(bool, copy=True)
        plane = self.planes[row].astype(np.int64, copy=True)
        plane[~mask] = 0
        return {
            "features": torch.from_numpy(self.features[row].astype(np.float32)),
            "planes": torch.from_numpy(plane),
            "mask": torch.from_numpy(mask),
            "targets": torch.from_numpy(self.targets[row].astype(np.float32)),
            "weights": torch.from_numpy(self.weights[row].astype(np.float32)),
            "row": row,
        }


# Contract-mode dataset regression catches padding/mask alignment before GPU work.
_contract_features = np.zeros((2, 4, 384), dtype=np.float16)
_contract_planes = np.array([[0, 1, 2, -1], [2, 2, -1, -1]], dtype=np.int8)
_contract_masks = _contract_planes >= 0
_contract_targets = np.zeros((2, 12), dtype=np.float32)
_contract_weights = np.ones((2, 12), dtype=np.float32)
_contract_dataset = CachedFeatureDataset(
    _contract_features,
    _contract_planes,
    _contract_masks,
    _contract_targets,
    _contract_weights,
    np.array([0, 1]),
)
assert _contract_dataset[0]["planes"].tolist() == [0, 1, 2, 0]
assert _contract_dataset[0]["mask"].tolist() == [True, True, True, False]


def extract_study_feature_cache(
    study_frame: pd.DataFrame,
    series_frame: pd.DataFrame,
    image_root: Path,
    slices_per_plane: int,
    cache_label: str,
) -> dict[str, np.ndarray]:
    if slices_per_plane not in (2, 8):
        raise ValueError("slices_per_plane must be 2 for smoke or 8 for full")
    cache_root = Path("/kaggle/working") / f"cache_{RUN_NONCE}" / "features"
    cache_root.mkdir(parents=True, exist_ok=True)
    cache_path = cache_root / f"{cache_label}_{slices_per_plane}.npz"
    maximum_instances = 3 * slices_per_plane
    expected_shape = (len(study_frame), maximum_instances, 384)

    if cache_path.exists():
        loaded = np.load(cache_path, allow_pickle=False)
        payload = {name: loaded[name] for name in loaded.files}
        loaded.close()
        cache_valid = (
            payload.get("features", np.empty(0)).shape == expected_shape
            and payload.get("planes", np.empty(0)).shape == expected_shape[:2]
            and payload.get("masks", np.empty(0)).shape == expected_shape[:2]
            and payload["features"].dtype == np.float16
            and payload["planes"].dtype == np.int8
            and payload["masks"].dtype == np.bool_
            and np.isfinite(payload["features"]).all()
            and np.all(
                (payload["planes"][payload["masks"]] >= 0)
                & (payload["planes"][payload["masks"]] < 3)
            )
            and np.all(payload["planes"][~payload["masks"]] == -1)
            and np.all(payload["features"][~payload["masks"]] == 0)
            and np.all(payload["masks"].sum(axis=1) <= maximum_instances)
        )
        if cache_valid:
            return payload
        raise RuntimeError("run-private cache integrity check failed")

    features = np.zeros(expected_shape, dtype=np.float16)
    planes = np.full(expected_shape[:2], -1, dtype=np.int8)
    masks = np.zeros(expected_shape[:2], dtype=bool)
    decode_failures = 0
    missing_planes = 0
    fallback_studies = 0
    no_series_studies = 0
    study_seconds = []
    grouped_series = {
        str(study_id): rows
        for study_id, rows in series_frame.groupby(ID_COLUMN, sort=False)
    }

    extraction_started = time.time()
    for output_row, study_id in enumerate(study_frame[ID_COLUMN].astype(str).tolist()):
        study_started = time.time()
        rows = grouped_series.get(study_id)
        slice_batches = []
        plane_values = []
        chosen_planes = set()
        if rows is not None:
            chosen = choose_plane_series(rows)
            for _, series_row in chosen.iterrows():
                plane_name = str(series_row["Anatomical_Plane"]).strip()
                if plane_name not in PLANE_TO_INDEX:
                    continue
                plane_index = PLANE_TO_INDEX[plane_name]
                series_directory = image_root / study_id / str(series_row["SeriesInstanceUID"])
                try:
                    sampled, _, diagnostics = sample_series_slices(
                        series_directory,
                        count=slices_per_plane,
                        image_size=224,
                    )
                    slice_batches.append(sampled)
                    plane_values.extend([plane_index] * len(sampled))
                    chosen_planes.add(plane_index)
                    decode_failures += int(diagnostics.get("decode_failures", 0))
                    decode_failures += int(diagnostics.get("multiframe_rejected", 0))
                except (InvalidDicomSlice, UnsupportedMultiframe, OSError, ValueError):
                    decode_failures += 1

        else:
            no_series_studies += 1
        missing_planes += 3 - len(chosen_planes)
        if slice_batches:
            images = np.concatenate(slice_batches, axis=0)
            embeddings = extract_dino_embeddings(images, batch_size=64)
            count = len(embeddings)
            if count > maximum_instances:
                raise RuntimeError("study exceeded fixed cache capacity")
            features[output_row, :count] = embeddings
            planes[output_row, :count] = np.asarray(plane_values, dtype=np.int8)
            masks[output_row, :count] = True
        else:
            fallback_studies += 1

        study_seconds.append(time.time() - study_started)
        completed = output_row + 1
        if completed % 100 == 0 or completed == len(study_frame):
            elapsed = time.time() - extraction_started
            rate = completed / max(elapsed, 1e-6)
            projected_hours = (
                np.quantile(study_seconds, 0.75) * len(study_frame) / 3600.0
                + 0.50
            )
            print(
                f"  feature progress: {completed}/{len(study_frame)}, "
                f"rate={rate:.2f} studies/s, projected={projected_hours:.2f}h"
            )
            if (
                RUN_MODE == "full"
                and completed >= 200
                and projected_hours > FULL_RUNTIME_ABORT_HOURS
            ):
                raise RuntimeError(
                    f"Projected feature runtime {projected_hours:.2f}h exceeds "
                    f"{FULL_RUNTIME_ABORT_HOURS:.1f}h gate"
                )

    if not np.isfinite(features).all():
        raise RuntimeError("non-finite values in feature cache")
    if np.any(masks.sum(axis=1) > maximum_instances):
        raise RuntimeError("invalid feature mask")
    if np.any(planes[~masks] != -1) or np.any(features[~masks] != 0):
        raise RuntimeError("feature-cache padding contract failed")

    temporary_path = cache_path.with_suffix(".tmp")
    with temporary_path.open("wb") as handle:
        np.savez_compressed(
            handle,
            features=features,
            planes=planes,
            masks=masks,
            decode_failures=np.array([decode_failures], dtype=np.int64),
            missing_planes=np.array([missing_planes], dtype=np.int64),
            fallback_studies=np.array([fallback_studies], dtype=np.int64),
            no_series_studies=np.array([no_series_studies], dtype=np.int64),
            elapsed_seconds=np.array([time.time() - extraction_started], dtype=np.float64),
        )
    os.replace(temporary_path, cache_path)
    return {
        "features": features,
        "planes": planes,
        "masks": masks,
        "decode_failures": np.array([decode_failures], dtype=np.int64),
        "missing_planes": np.array([missing_planes], dtype=np.int64),
        "fallback_studies": np.array([fallback_studies], dtype=np.int64),
        "no_series_studies": np.array([no_series_studies], dtype=np.int64),
        "elapsed_seconds": np.array([time.time() - extraction_started], dtype=np.float64),
    }


def loader_for(
    payload: dict[str, np.ndarray],
    targets: np.ndarray,
    weights: np.ndarray,
    row_indices: np.ndarray,
    shuffle: bool,
    batch_size: int,
    seed: int,
) -> DataLoader:
    dataset = CachedFeatureDataset(
        payload["features"],
        payload["planes"],
        payload["masks"],
        targets,
        weights,
        row_indices,
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=bool(torch.cuda.is_available()),
        generator=generator,
    )


def auc_summary(targets: np.ndarray, predictions: np.ndarray) -> tuple[float, int]:
    scores = []
    for target_index in range(targets.shape[1]):
        known = np.isfinite(targets[:, target_index])
        truth = targets[known, target_index]
        estimate = predictions[known, target_index]
        if len(truth) and len(np.unique(truth)) == 2:
            scores.append(roc_auc_score(truth, estimate))
    return (float(np.mean(scores)) if scores else float("nan"), len(scores))


def train_fold_model(
    payload: dict[str, np.ndarray],
    targets: np.ndarray,
    weights: np.ndarray,
    training_rows: np.ndarray,
    validation_rows: np.ndarray,
    epochs: int,
    patience: int,
    seed: int,
):
    valid_images = payload["masks"].any(axis=1)
    supervised = (np.nan_to_num(weights) > 0).any(axis=1)
    training_rows = np.asarray([
        row for row in training_rows if valid_images[row] and supervised[row]
    ], dtype=np.int64)
    validation_rows = np.asarray([
        row for row in validation_rows if valid_images[row]
    ], dtype=np.int64)
    if len(training_rows) == 0 or len(validation_rows) == 0:
        raise RuntimeError("fold has no usable train or validation studies")

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TargetAttentionMIL(dropout=0.20).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=3e-4, weight_decay=1e-3
    )
    positive_weight = fold_positive_weight(
        targets[training_rows], weights[training_rows]
    ).to(device)

    train_loader = loader_for(
        payload, targets, weights, training_rows, True, 32, seed
    )
    valid_loader = loader_for(
        payload, targets, weights, validation_rows, False, 32, seed
    )

    best_state = None
    best_score = -np.inf
    best_loss = np.inf
    stale_epochs = 0
    for epoch in range(1, epochs + 1):
        model.train()
        training_losses = []
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(
                batch["features"].to(device, non_blocking=True),
                batch["planes"].to(device, non_blocking=True),
                batch["mask"].to(device, non_blocking=True),
            )
            loss = weighted_masked_bce(
                logits,
                batch["targets"].to(device, non_blocking=True),
                batch["weights"].to(device, non_blocking=True),
                positive_weight,
            )
            if not torch.isfinite(loss):
                raise RuntimeError("non-finite fold training loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            training_losses.append(float(loss.detach().cpu()))

        model.eval()
        validation_losses = []
        predictions = []
        truths = []
        with torch.inference_mode():
            for batch in valid_loader:
                logits = model(
                    batch["features"].to(device, non_blocking=True),
                    batch["planes"].to(device, non_blocking=True),
                    batch["mask"].to(device, non_blocking=True),
                )
                loss = weighted_masked_bce(
                    logits,
                    batch["targets"].to(device, non_blocking=True),
                    batch["weights"].to(device, non_blocking=True),
                    positive_weight,
                )
                validation_losses.append(float(loss.detach().cpu()))
                predictions.append(torch.sigmoid(logits).cpu().numpy())
                truths.append(batch["targets"].numpy())

        valid_predictions = np.concatenate(predictions, axis=0)
        valid_truths = np.concatenate(truths, axis=0)
        valid_loss = float(np.mean(validation_losses))
        macro_auc, scorable = auc_summary(valid_truths, valid_predictions)
        selection_score = (
            macro_auc
            if np.isfinite(macro_auc) and scorable >= 3
            else -valid_loss
        )
        if selection_score > best_score:
            best_score = selection_score
            best_loss = valid_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1
        print(
            f"  epoch={epoch}, train_loss={np.mean(training_losses):.4f}, "
            f"valid_loss={valid_loss:.4f}, scorable={scorable}"
        )
        if stale_epochs >= patience:
            break

    if best_state is None:
        raise RuntimeError("fold training did not produce a checkpoint")
    model.load_state_dict(best_state)
    model.to(device).eval()

    ordered_predictions = []
    with torch.inference_mode():
        for batch in valid_loader:
            logits = model(
                batch["features"].to(device, non_blocking=True),
                batch["planes"].to(device, non_blocking=True),
                batch["mask"].to(device, non_blocking=True),
            )
            ordered_predictions.append(torch.sigmoid(logits).cpu().numpy())
    return model, np.concatenate(ordered_predictions, axis=0), best_score, best_loss


print(
    "FOLD CONTRACT PASSED: "
    f"gold={len(gold_rows)}, folds={validation_sizes}, method={fold_method}, "
    f"patient_grouping={'enabled' if patient_column else 'unavailable'}"
)

TRAINING_FEATURE_PAYLOAD = None
FOLD_MODELS = []
FOLD_CHECKPOINT_PATHS = []
OOF_PREDICTIONS = None
OOF_ASSIGNMENT_COUNTS = None
OOF_MACRO_AUC = float("nan")
OOF_SCORABLE_TARGETS = 0

if RUN_MODE in {"smoke", "full"}:
    if not torch.cuda.is_available():
        raise RuntimeError(f"{RUN_MODE} cached-feature training requires a compatible T4")
    slices_per_plane = 2 if RUN_MODE == "smoke" else 8

    if RUN_MODE == "smoke":
        validation_global = gold_positions[folds[0][1]]
        training_gold_global = gold_positions[folds[0][0]][:6]
        weak_candidates = np.flatnonzero(
            (~gold_mask) & (np.nan_to_num(weak_weights).sum(axis=1) > 0)
        )
        needed_weak = 24 - len(validation_global) - len(training_gold_global)
        if needed_weak < 1 or len(weak_candidates) < needed_weak:
            raise RuntimeError("insufficient weak-label studies for smoke composition")
        rng = np.random.default_rng(FOLD_SEED)
        chosen_weak = np.sort(
            rng.choice(weak_candidates, size=needed_weak, replace=False)
        )
        selected_global = np.concatenate(
            [validation_global, training_gold_global, chosen_weak]
        )
        assert len(selected_global) == 24 and len(np.unique(selected_global)) == 24
    else:
        selected_global = np.arange(len(train_df), dtype=np.int64)

    selected_frame = train_df.iloc[selected_global].reset_index(drop=True)
    selected_targets = weak_values[selected_global].astype(np.float32, copy=True)
    selected_weights = weak_weights[selected_global].astype(np.float32, copy=True)
    official_selected = train_df.iloc[selected_global][
        TARGET_COLUMNS
    ].to_numpy(dtype=np.float32)
    official_known = np.isfinite(official_selected)
    selected_targets[official_known] = official_selected[official_known]
    selected_weights[official_known] = 1.0
    assert np.isin(selected_targets[official_known], [0.0, 1.0]).all()
    assert np.all(selected_weights[official_known] == 1.0)
    TRAINING_FEATURE_PAYLOAD = extract_study_feature_cache(
        selected_frame,
        train_series_df,
        TRAIN_IMAGE_ROOT,
        slices_per_plane,
        f"train_{RUN_MODE}",
    )
    fallback_count = int(TRAINING_FEATURE_PAYLOAD["fallback_studies"][0])
    allowed_fallbacks = 0 if RUN_MODE == "smoke" else max(
        5, int(np.ceil(0.01 * len(selected_frame)))
    )
    if fallback_count > allowed_fallbacks:
        raise RuntimeError(
            f"Feature coverage gate failed: fallbacks={fallback_count}, "
            f"allowed={allowed_fallbacks}"
        )

    if RUN_MODE == "smoke":
        validation_local = np.flatnonzero(
            np.isin(selected_global, validation_global)
        )
        training_local = np.flatnonzero(
            ~np.isin(selected_global, validation_global)
        )
        smoke_model, smoke_predictions, smoke_score, smoke_loss = train_fold_model(
            TRAINING_FEATURE_PAYLOAD,
            selected_targets,
            selected_weights,
            training_local,
            validation_local,
            epochs=2,
            patience=2,
            seed=FOLD_SEED,
        )
        assert smoke_predictions.shape == (len(validation_local), 12)
        assert np.isfinite(smoke_predictions).all()
        FOLD_MODELS = [smoke_model]
        smoke_fold_root = (
            Path("/kaggle/working") / f"cache_{RUN_NONCE}" / "folds"
        )
        smoke_fold_root.mkdir(parents=True, exist_ok=True)
        smoke_checkpoint_path = smoke_fold_root / "fold_0.pt"
        torch.save(
            {
                "run_nonce": RUN_NONCE,
                "fold": 0,
                "target_columns": TARGET_COLUMNS,
                "state_dict": {
                    key: value.detach().cpu().clone()
                    for key, value in smoke_model.state_dict().items()
                },
            },
            smoke_checkpoint_path,
        )
        FOLD_CHECKPOINT_PATHS = [smoke_checkpoint_path]
        print(
            "CACHED-FEATURE SMOKE PASSED: "
            f"studies={len(selected_frame)}, validation={len(validation_local)}, "
            f"instances={int(TRAINING_FEATURE_PAYLOAD['masks'].sum())}, "
            f"fallbacks={fallback_count}"
        )
    else:
        OOF_PREDICTIONS = np.full((len(gold_rows), 12), np.nan, dtype=np.float32)
        OOF_ASSIGNMENT_COUNTS = np.zeros(len(gold_rows), dtype=np.int8)
        cache_root = Path("/kaggle/working") / f"cache_{RUN_NONCE}" / "folds"
        cache_root.mkdir(parents=True, exist_ok=True)
        for fold_index, (gold_train_index, gold_valid_index) in enumerate(folds):
            validation_global = gold_positions[gold_valid_index]
            training_global = np.flatnonzero(
                ~np.isin(np.arange(len(train_df)), validation_global)
            )
            fold_model, fold_predictions, fold_score, fold_loss = train_fold_model(
                TRAINING_FEATURE_PAYLOAD,
                selected_targets,
                selected_weights,
                training_global,
                validation_global,
                epochs=20,
                patience=4,
                seed=FOLD_SEED + fold_index,
            )
            OOF_PREDICTIONS[gold_valid_index] = fold_predictions
            OOF_ASSIGNMENT_COUNTS[gold_valid_index] += 1
            checkpoint_path = cache_root / f"fold_{fold_index}.pt"
            torch.save(
                {
                    "run_nonce": RUN_NONCE,
                    "fold": fold_index,
                    "target_columns": TARGET_COLUMNS,
                    "state_dict": {
                        key: value.detach().cpu()
                        for key, value in fold_model.state_dict().items()
                    },
                },
                checkpoint_path,
            )
            FOLD_MODELS.append(fold_model)
            FOLD_CHECKPOINT_PATHS.append(checkpoint_path)
        assert len(FOLD_MODELS) == N_FOLDS
        assert len(FOLD_CHECKPOINT_PATHS) == N_FOLDS
        assert OOF_PREDICTIONS.shape == (58, 12)
        assert np.isfinite(OOF_PREDICTIONS).all()
        assert np.all(OOF_ASSIGNMENT_COUNTS == 1)
        OOF_MACRO_AUC, OOF_SCORABLE_TARGETS = auc_summary(
            gold_y, OOF_PREDICTIONS
        )
        if OOF_SCORABLE_TARGETS < 10 or not OOF_MACRO_AUC > 0.5833:
            raise RuntimeError(
                f"OOF performance gate failed: macro_auc={OOF_MACRO_AUC:.4f}, "
                f"scorable={OOF_SCORABLE_TARGETS}"
            )
        print(
            "FULL FIVE-FOLD TRAINING PASSED: "
            f"oof_rows=58, macro_auc={OOF_MACRO_AUC:.4f}, "
            f"scorable={OOF_SCORABLE_TARGETS}, fallbacks={fallback_count}"
        )


# RSNA Knee Abnormality Detection — Report-Supervised DINOv2 MIL

## Purpose

This private competition notebook predicts all 12 knee abnormality targets from multi-plane MRI studies. It is a research system for the Kaggle competition, not a clinical diagnostic tool.

## Pipeline

1. **Current-run contract** validates the five competition CSV files, exact target order, unique study keys, run mode, and a fresh run nonce.
2. **Report supervision** converts the report-only training studies into confidence-weighted positive, negative, or unknown labels. It uses bounded negation, uncertainty, and history rules without printing report text or identifiers.
3. **MRI sampling** deterministically selects one series per sagittal, coronal, and axial plane; physically orders DICOM slices; normalizes intensities; and samples the central anatomy.
4. **Frozen DINOv2 encoder** extracts one 384-dimensional feature per slice from the attached Kaggle-hosted ViT-S/14 model. Features, plane indices, and masks are cached only under the current run nonce.
5. **Target-aware MIL** uses 12 learned queries so each abnormality can attend to different slices. Official labels have weight 1.0; report-derived labels use lower confidence weights.
6. **Leakage-safe validation** assigns all 58 official studies to five deterministic folds exactly once. A held-out gold study is excluded from every training path, including its report label.
7. **Inference** rank-averages fold predictions, writes the exact runtime sample-submission schema atomically, then reloads and independently validates the CSV and SHA-256.

## Run modes and safety gates

- contract: CPU checks only.
- smoke: T4 x2, 24 training studies, two slices per plane, one fold, and visible-test validation.
- full: T4 x2, all training studies, eight slices per plane, five folds, and a 7.5-hour projected-runtime abort.

The full path stops on incompatible CUDA hardware, systemic image failures, non-finite values, fold leakage, fewer than 10 scorable targets, OOF macro AUC at or below 0.5833, stale output, or a schema mismatch. The notebook does not print reports, study identifiers, DICOM pixels, or submission rows.

## Attached resources

- Competition: RSNA Knee Abnormality Detection
- Encoder: Meta Research DINOv2 small, Kaggle Model input metaresearch/dinov2/pytorch/small/1
- Internet is not required during execution.


In [ ]:
# ============================================================
# Rank-averaged inference and independent submission validation
# ============================================================
import hashlib
import json
import os
from pathlib import Path


def rank_average(fold_predictions: list[np.ndarray]) -> np.ndarray:
    if not fold_predictions:
        raise ValueError("at least one fold prediction array is required")
    reference_shape = np.asarray(fold_predictions[0]).shape
    if len(reference_shape) != 2 or reference_shape[1] != len(TARGET_COLUMNS):
        raise ValueError("fold predictions must have shape [study, target]")
    for prediction in fold_predictions:
        values = np.asarray(prediction)
        if values.shape != reference_shape:
            raise ValueError("fold prediction shapes are inconsistent")
        if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
            raise ValueError("fold predictions must be finite probabilities")
    stacked = np.stack(fold_predictions, axis=0).astype(np.float64)
    ranked = np.empty_like(stacked, dtype=np.float64)
    for fold_index in range(stacked.shape[0]):
        for target_index in range(stacked.shape[2]):
            ranked[fold_index, :, target_index] = pd.Series(
                stacked[fold_index, :, target_index]
            ).rank(method="average", pct=True).to_numpy()
    return ranked.mean(axis=0).clip(1e-5, 1.0 - 1e-5)


_rank_test_a = np.tile(np.array([[0.1], [0.5], [0.9]]), (1, 12))
_rank_test_b = np.tile(np.array([[0.5], [0.1], [0.9]]), (1, 12))
_rank_result = rank_average([_rank_test_a, _rank_test_b])
assert _rank_result.shape == (3, 12)
assert np.allclose(_rank_result[:, 0], np.array([0.5, 0.5, 1.0 - 1e-5]))
assert np.allclose(
    rank_average([_rank_test_a]),
    rank_average([_rank_test_a ** 3]),
)
assert np.isfinite(_rank_result).all()
assert np.all((_rank_result > 0) & (_rank_result < 1))


def supervision_prevalence(excluded_rows: np.ndarray) -> np.ndarray:
    included = np.ones(len(train_df), dtype=bool)
    included[np.asarray(excluded_rows, dtype=np.int64)] = False
    fold_values = weak_values[included]
    fold_weights = weak_weights[included]
    known = (
        np.isfinite(fold_values)
        & np.isfinite(fold_weights)
        & (fold_weights > 0)
    )
    values = np.where(known, fold_values, 0.0)
    weights = np.where(known, fold_weights, 0.0)
    numerator = (values * weights).sum(axis=0)
    denominator = weights.sum(axis=0)
    fallback = np.divide(
        numerator,
        denominator,
        out=np.full(len(TARGET_COLUMNS), 0.5, dtype=np.float64),
        where=denominator > 0,
    )
    return np.clip(fallback, 1e-5, 1.0 - 1e-5)


def predict_cached_studies(
    model: torch.nn.Module,
    payload: dict[str, np.ndarray],
    fallback_values: np.ndarray,
) -> np.ndarray:
    features = payload["features"]
    planes = payload["planes"]
    masks = payload["masks"]
    valid_rows = np.flatnonzero(masks.any(axis=1))
    output = np.full(
        (len(features), len(TARGET_COLUMNS)), np.nan, dtype=np.float64
    )
    invalid_rows = np.flatnonzero(~masks.any(axis=1))
    output[invalid_rows] = fallback_values
    if len(valid_rows) == 0:
        if not np.isfinite(output).all():
            raise RuntimeError("zero-valid fallback predictions are incomplete")
        return output

    dummy_targets = np.zeros((len(features), len(TARGET_COLUMNS)), dtype=np.float32)
    dummy_weights = np.zeros_like(dummy_targets)
    loader = loader_for(
        payload,
        dummy_targets,
        dummy_weights,
        valid_rows,
        shuffle=False,
        batch_size=32,
        seed=FOLD_SEED,
    )
    device = next(model.parameters()).device
    model.eval()
    prediction_batches = []
    with torch.inference_mode():
        for batch in loader:
            logits = model(
                batch["features"].to(device, non_blocking=True),
                batch["planes"].to(device, non_blocking=True),
                batch["mask"].to(device, non_blocking=True),
            )
            probabilities = torch.sigmoid(logits).float().cpu().numpy()
            if not np.isfinite(probabilities).all():
                raise RuntimeError("non-finite inference probabilities")
            prediction_batches.append(probabilities)
    valid_predictions = np.concatenate(prediction_batches, axis=0)
    if valid_predictions.shape != (len(valid_rows), len(TARGET_COLUMNS)):
        raise RuntimeError("valid-row inference order/count contract failed")
    output[valid_rows] = valid_predictions
    if not np.isfinite(output).all() or np.any((output < 0) | (output > 1)):
        raise RuntimeError("fold inference left invalid or unfilled predictions")
    return output


def atomic_write_submission(
    predictions: np.ndarray,
    fallback_count: int,
    elapsed_seconds: float,
) -> tuple[Path, Path, str]:
    if predictions.shape != (len(sample_submission_df), len(TARGET_COLUMNS)):
        raise ValueError("prediction matrix does not match runtime sample submission")
    if not np.isfinite(predictions).all():
        raise ValueError("submission predictions are non-finite")
    if np.any((predictions < 0) | (predictions > 1)):
        raise ValueError("submission predictions are outside [0, 1]")

    working_root = Path("/kaggle/working")
    nonce_submission = working_root / f"submission_{RUN_NONCE}.csv"
    manifest_path = working_root / f"run_{RUN_NONCE}.json"
    final_submission = working_root / "submission.csv"
    temporary_submission = working_root / f".submission_{RUN_NONCE}.tmp"

    submission = sample_submission_df.copy()
    submission.loc[:, TARGET_COLUMNS] = predictions
    submission.to_csv(temporary_submission, index=False)
    os.replace(temporary_submission, nonce_submission)
    submission_sha256 = hashlib.sha256(nonce_submission.read_bytes()).hexdigest()

    manifest = {
        "run_nonce": RUN_NONCE,
        "mode": RUN_MODE,
        "submission_rows": int(len(submission)),
        "submission_columns": int(submission.shape[1]),
        "target_count": int(len(TARGET_COLUMNS)),
        "fold_count": int(len(FOLD_MODELS)),
        "slices_per_plane": int(2 if RUN_MODE == "smoke" else 8),
        "fallback_count": int(fallback_count),
        "elapsed_seconds": float(elapsed_seconds),
        "oof_macro_auc": (
            float(OOF_MACRO_AUC) if np.isfinite(OOF_MACRO_AUC) else None
        ),
        "oof_scorable_targets": int(OOF_SCORABLE_TARGETS),
        "sha256": submission_sha256,
    }
    temporary_manifest = manifest_path.with_suffix(".tmp")
    with temporary_manifest.open("w", encoding="utf-8") as handle:
        json.dump(
            manifest, handle, sort_keys=True, indent=2, allow_nan=False
        )
    os.replace(temporary_manifest, manifest_path)

    temporary_final = working_root / f".submission_current_{RUN_NONCE}.tmp"
    temporary_final.write_bytes(nonce_submission.read_bytes())
    os.replace(temporary_final, final_submission)
    return final_submission, manifest_path, submission_sha256


def independently_validate_submission(
    submission_path: Path,
    manifest_path: Path,
) -> tuple[pd.DataFrame, dict, str]:
    nonce_path = Path("/kaggle/working") / f"submission_{RUN_NONCE}.csv"
    if not submission_path.exists() or not manifest_path.exists() or not nonce_path.exists():
        raise RuntimeError("current-run submission artifacts are missing")
    run_started_ns = int(RUN_STARTED_AT * 1_000_000_000)
    for artifact in (submission_path, manifest_path, nonce_path):
        if artifact.stat().st_mtime_ns < run_started_ns:
            raise RuntimeError("submission artifact predates the current run")
    verified = pd.read_csv(submission_path)
    with manifest_path.open("r", encoding="utf-8") as handle:
        manifest = json.load(handle)
    checksum = hashlib.sha256(submission_path.read_bytes()).hexdigest()
    nonce_checksum = hashlib.sha256(nonce_path.read_bytes()).hexdigest()

    assert manifest["run_nonce"] == RUN_NONCE
    assert manifest["mode"] == RUN_MODE
    assert manifest["sha256"] == checksum == nonce_checksum
    assert manifest["submission_rows"] == len(sample_submission_df)
    assert manifest["submission_columns"] == sample_submission_df.shape[1]
    assert manifest["fold_count"] == len(FOLD_MODELS)
    assert manifest["fallback_count"] >= 0
    assert list(verified.columns) == list(sample_submission_df.columns)
    assert verified.shape == sample_submission_df.shape
    assert verified[ID_COLUMN].astype(str).tolist() == (
        sample_submission_df[ID_COLUMN].astype(str).tolist()
    )
    assert verified[ID_COLUMN].is_unique
    verified_values = verified[TARGET_COLUMNS].to_numpy(dtype=np.float64)
    assert np.isfinite(verified_values).all()
    assert np.all((verified_values >= 0) & (verified_values <= 1))
    assert not any(str(column).lower().startswith("unnamed") for column in verified.columns)
    assert manifest["submission_rows"] == len(verified)
    assert manifest["target_count"] == len(TARGET_COLUMNS)
    return verified, manifest, checksum


print("INFERENCE CONTRACT PASSED: rank averaging and stale-output guards are valid")

FINAL_SUBMISSION_PATH = None
FINAL_MANIFEST_PATH = None
FINAL_SUBMISSION_SHA256 = None

if RUN_MODE in {"smoke", "full"}:
    if not torch.cuda.is_available():
        raise RuntimeError(f"{RUN_MODE} inference requires a compatible T4")
    expected_models = 1 if RUN_MODE == "smoke" else N_FOLDS
    if len(FOLD_MODELS) != expected_models:
        raise RuntimeError(
            f"expected {expected_models} trained fold models, found {len(FOLD_MODELS)}"
        )
    if len(FOLD_CHECKPOINT_PATHS) != expected_models:
        raise RuntimeError("mode-specific fold checkpoint count is invalid")
    nonce_cache_root = (
        Path("/kaggle/working") / f"cache_{RUN_NONCE}"
    ).resolve()
    for model_index, checkpoint_path in enumerate(FOLD_CHECKPOINT_PATHS):
        resolved_checkpoint = Path(checkpoint_path).resolve()
        if (
            not resolved_checkpoint.exists()
            or not str(resolved_checkpoint).startswith(str(nonce_cache_root) + os.sep)
        ):
            raise RuntimeError("fold checkpoint is outside the current nonce cache")
        checkpoint = torch.load(
            resolved_checkpoint, map_location="cpu", weights_only=False
        )
        if (
            checkpoint.get("run_nonce") != RUN_NONCE
            or int(checkpoint.get("fold", -1)) != model_index
            or checkpoint.get("target_columns") != TARGET_COLUMNS
        ):
            raise RuntimeError("fold checkpoint metadata mismatch")

    inference_started = time.time()
    TEST_IMAGE_ROOT = DATA_ROOT / "test_series"
    test_slices_per_plane = 2 if RUN_MODE == "smoke" else 8
    TEST_FEATURE_PAYLOAD = extract_study_feature_cache(
        test_df,
        test_series_df,
        TEST_IMAGE_ROOT,
        test_slices_per_plane,
        f"test_{RUN_MODE}",
    )
    test_fallback_count = int(TEST_FEATURE_PAYLOAD["fallback_studies"][0])
    if RUN_MODE == "smoke" and test_fallback_count != 0:
        raise RuntimeError("smoke test inference produced a fallback study")
    if RUN_MODE == "full" and test_fallback_count > max(
        2, int(np.ceil(0.01 * len(test_df)))
    ):
        raise RuntimeError("test feature coverage gate failed")

    per_fold_predictions = []
    for fold_index, model in enumerate(FOLD_MODELS):
        excluded_gold_rows = gold_positions[folds[fold_index][1]]
        fold_fallback_values = supervision_prevalence(excluded_gold_rows)
        fold_predictions = predict_cached_studies(
            model, TEST_FEATURE_PAYLOAD, fold_fallback_values
        )
        if (
            fold_predictions.shape != (len(test_df), len(TARGET_COLUMNS))
            or not np.isfinite(fold_predictions).all()
            or np.any((fold_predictions < 0) | (fold_predictions > 1))
        ):
            raise RuntimeError("fold prediction contract failed")
        per_fold_predictions.append(fold_predictions)
    final_predictions = rank_average(per_fold_predictions)
    inference_elapsed = time.time() - inference_started

    (
        FINAL_SUBMISSION_PATH,
        FINAL_MANIFEST_PATH,
        FINAL_SUBMISSION_SHA256,
    ) = atomic_write_submission(
        final_predictions,
        test_fallback_count,
        time.time() - RUN_STARTED_AT,
    )
    verified_submission, verified_manifest, verified_checksum = (
        independently_validate_submission(
            FINAL_SUBMISSION_PATH,
            FINAL_MANIFEST_PATH,
        )
    )
    verified_values = verified_submission[TARGET_COLUMNS].to_numpy(dtype=np.float64)
    print(
        "SUBMISSION VALIDATION PASSED: "
        f"shape={verified_submission.shape}, "
        f"min={verified_values.min():.6f}, "
        f"max={verified_values.max():.6f}, "
        f"mean={verified_values.mean():.6f}, "
        f"fallbacks={test_fallback_count}, "
        f"inference_seconds={inference_elapsed:.1f}, "
        f"sha256={verified_checksum}"
    )
